In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
chars=sorted(list(set(text)))
vocab=len(chars)
print(''.join(chars))
vocab


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


65

In [5]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
encode=lambda s: [stoi[c] for c in s]
decode= lambda d: ''.join([itos[c]for c in d])
print(encode("hello"))
decode(encode("hello"))

[46, 43, 50, 50, 53]


'hello'

In [6]:
import torch
data=torch.tensor(encode(text),dtype=torch.long)
data.shape,data.type

(torch.Size([1115394]), <function Tensor.type>)

In [7]:
n=int(0.9*len(data))
train_data=data[:n]
test_data=data[n:]
train_data.shape,test_data.shape

(torch.Size([1003854]), torch.Size([111540]))

In [8]:
torch.manual_seed(1337)
context_window=8
batch_size=4
def get_batch(split):
    data=train_data if split=='train' else test_data
    ix=torch.randint(len(data)-context_window,(batch_size,))
    x=torch.stack([data[i:i+context_window] for i in ix])
    y=torch.stack([data[i:i+context_window+1] for i in ix])
    return x,y
xb,yb=get_batch('train')
print('inputs:')
print(xb.shape,yb.shape)
xb,yb
    


inputs:
torch.Size([4, 8]) torch.Size([4, 9])


(tensor([[24, 43, 58,  5, 57,  1, 46, 43],
         [44, 53, 56,  1, 58, 46, 39, 58],
         [52, 58,  1, 58, 46, 39, 58,  1],
         [25, 17, 27, 10,  0, 21,  1, 54]]),
 tensor([[24, 43, 58,  5, 57,  1, 46, 43, 39],
         [44, 53, 56,  1, 58, 46, 39, 58,  1],
         [52, 58,  1, 58, 46, 39, 58,  1, 46],
         [25, 17, 27, 10,  0, 21,  1, 54, 39]]))

In [23]:
torch.manual_seed(42)

x=torch.randn(B,T,C)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[7., 3.],
        [1., 8.],
        [6., 4.]])
--
c=
tensor([[7.0000, 3.0000],
        [4.0000, 5.5000],
        [4.6667, 5.0000]])


In [37]:
B,T,C=4,8,32
x=torch.randn(B,T,C)
head_size=16
query=nn.Linear(C,head_size,bias=False)
key=nn.Linear(C,head_size,bias=False)
value=nn.Linear(C,head_size,bias=False)
k=key(x)
q=query(x)
wei=q@k.transpose(-2,-1)
tril=torch.tril(torch.ones(T,T))
wei=wei.masked_fill(tril==0,float('-inf'))
wei=F.softmax(wei,dim=-1)
v=value(x)
out=wei@v
#out=wei@x
out.shape

torch.Size([4, 8, 16])

In [40]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.8541, 0.1459, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2679, 0.7021, 0.0300, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4869, 0.2241, 0.1370, 0.1521, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2171, 0.3107, 0.2411, 0.1543, 0.0768, 0.0000, 0.0000, 0.0000],
        [0.1103, 0.3312, 0.0741, 0.1064, 0.0854, 0.2927, 0.0000, 0.0000],
        [0.1511, 0.1672, 0.0456, 0.4244, 0.0322, 0.0516, 0.1279, 0.0000],
        [0.0462, 0.7764, 0.0159, 0.0389, 0.0096, 0.0834, 0.0220, 0.0075]],
       grad_fn=<SelectBackward0>)